# STEP 3: Using Tools
Make models to fetch data and take actions

Docs: https://developers.openai.com/api/docs/guides/function-calling

In [2]:
import json
import os

import requests
from openai import OpenAI
from pydantic import BaseModel, Field

In [14]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [3]:
def get_weather(latitude, longitude):
    response = requests.get(
        f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m"
    )
    data = response.json()
    return data["current"]

In [12]:
# call model with get_weather tool defined
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get weather for provided latitude and longitude in celcius",
            "parameters": {
                "type": "object",
                "properties": {
                    "latitude": { "type": "number" },
                    "longitude": { "type": "number" },
                },
                "required": ["latitude", "longitude"],
                "additionalProperties": False,
            },
            "strict": True,
        },
    }
]

In [5]:
system_prompt = "You are a helpful weather assistant"

In [27]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "What's the weather like in Paris today?"},
]

In [28]:
completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    tools=tools
)

In [29]:
completion.model_dump()

{'id': 'chatcmpl-DdInaflEriVSz6hjFDbIvDQ9d5GN9',
 'choices': [{'finish_reason': 'tool_calls',
   'index': 0,
   'logprobs': None,
   'message': {'content': None,
    'refusal': None,
    'role': 'assistant',
    'annotations': [],
    'audio': None,
    'function_call': None,
    'tool_calls': [{'id': 'call_mfbm4kbrQHKrnMCYmJM8EseX',
      'function': {'arguments': '{"latitude":48.8566,"longitude":2.3522}',
       'name': 'get_weather'},
      'type': 'function'}]}}],
 'created': 1778259426,
 'model': 'gpt-4o-mini-2024-07-18',
 'object': 'chat.completion',
 'service_tier': 'default',
 'system_fingerprint': 'fp_5d26835d99',
 'usage': {'completion_tokens': 24,
  'prompt_tokens': 67,
  'total_tokens': 91,
  'completion_tokens_details': {'accepted_prediction_tokens': 0,
   'audio_tokens': 0,
   'reasoning_tokens': 0,
   'rejected_prediction_tokens': 0},
  'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}}

In [30]:
def call_function(name, args):
    if name == "get_weather":
        return get_weather(**args)

In [31]:
for tool_call in completion.choices[0].message.tool_calls:
    name = tool_call.function.name
    args = json.loads(tool_call.function.arguments)
    messages.append(completion.choices[0].message)

    result = call_function(name, args)
    messages.append(
        {"role": "tool", "tool_call_id": tool_call.id, "content": json.dumps(result)}
    )

In [32]:
messages

[{'role': 'system', 'content': 'You are a helpful weather assistant'},
 {'role': 'user', 'content': "What's the weather like in Paris today?"},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_mfbm4kbrQHKrnMCYmJM8EseX', function=Function(arguments='{"latitude":48.8566,"longitude":2.3522}', name='get_weather'), type='function')]),
 {'role': 'tool',
  'tool_call_id': 'call_mfbm4kbrQHKrnMCYmJM8EseX',
  'content': '{"time": "2026-05-08T16:45", "interval": 900, "temperature_2m": 19.9, "wind_speed_10m": 10.0}'}]

In [33]:
class WeatherResponseFormat(BaseModel):
    temperature: float = Field(
        description="The current temperature in celsius for the given location."
    )
    reponse: str = Field(
        description="A natural language response to the user's question."
    )

In [34]:
completion_2 = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=messages,
    tools=tools,
    response_format=WeatherResponseFormat,
)

In [36]:
final_response = completion_2.choices[0].message.parsed
final_response.temperature
final_response.reponse

'Today in Paris, the weather is pleasant with a temperature of about 19.9°C. Expect some wind with a speed of 10.0 km/h.'